In [1]:
from z_matching import *
import numpy as np
from astropy.table import Table
from wlclusters import *
from astropy.cosmology import Planck18 as cosmo
import os
os.chdir(r"/local/home/ib286534/Documents/Catalogues")

## Lecture des catalogues

In [2]:
# catalogue de sources (RR2 Euclid)
source_cat = Table.read('cosmohub_RR2_lensmc_v1.4_21987_noNaN.fits', format='fits', memmap=True)
source_cat.rename_columns(['SHE_RA', 'SHE_DEC', 'SHE_E1_CORRECTED', 'SHE_E2_CORRECTED', 'SHE_WEIGHT', 'PHZ_MEDIAN'],
       ['RA', 'Dec', 'e_1', 'e_2', 'weight', 'z_p'])

In [3]:
# catalogue de detections avec redshifts (eROSITA)
erass = Table.read("large_erass_sample.fits")

# estimation du SNR de eRASS à partir de la masse et du redshift
erass_snr = estimateCatSNR(erass["BEST_Z"], erass["M500"] * 1e13, source_cat["z_p"], delta=500)
erass.add_column(erass_snr, name="WL_SNR")

In [4]:
## detections optiques (TR1 Euclid)
# tr1 = Table.read("unified_clusters_gluematch_direct_DETonly_FILTERED_0.6Mpc_20260224_1624462026-02-24T16_24_56.fits")
## Ne garder qu'une méthode de détection pour avoir des SNR comparables 
# tr1 = tr1[tr1["DET_CODE_NB"] == 2]

In [5]:
# catalogue de toutes les detections weak lensing (Euclid, échelle 4')
wl_cat = Table.read('all_euclid_detections_noise.fits', format='fits', memmap=True)
# On ne garde que les détections à haut SNR et on réindexe les ID
wl_cat = wl_cat[wl_cat["SNR"] > 4.5]
wl_cat.replace_column('ID', np.arange(len(wl_cat), dtype=int))

## Matching redshift

In [6]:
wl_cols = ['RA', 'Dec', 'SNR']

# Matching avec le catalogue optique TR1
# z_cols = ['RIGHT_ASCENSION_CLUSTER', 'DECLINATION_CLUSTER', "SNR_CLUSTER", 'Z_CLUSTER']
# matched_z_p = matchZ(wl_cat, tr1, healpix2rad(2048, 2.3), wl_cols, z_cols)

# Matching avec les detections eROSITA
z_cols = ['RA', 'DEC', 'WL_SNR', "BEST_Z"]
matched_z_p = matchZ(wl_cat, erass, healpix2rad(2048, 2.3), wl_cols, z_cols)

In [7]:
# Ajoute une colonne de redshift au catalogue et crée une version filtrée pour l'estimation de masse
wl_cat.add_column(matched_z_p, name='z_p')
wl_cat_matched = wl_cat[np.isfinite(matched_z_p)]

In [8]:
# Définition des bins radiaux pour l'extraction des profils de shear
bin_edges = np.logspace(np.log10(0.3/cosmo.h), np.log10(3.0/cosmo.h), 9)
# Extraction des profils de shear
shear_profiles = shear_extraction(cluster_cat=wl_cat_matched[:5],
                                  sources=source_cat, 
                                  bin_edges=bin_edges,
                                  dz=0.1,
                                  cosmo = cosmo)

Started pixelisation
Finished pixelisation


Processing Clusters: 100%|██████████| 5/5 [00:08<00:00,  1.72s/it]


In [9]:
# Estimation MCMC de la mass à partir des profils
all_chains, results = run(cluster_cat=wl_cat_matched[:5],
                 shear_profiles=shear_profiles,
                 parnames=['log10cdelt', 'log10mdelt'],
                 cosmo=cosmo,
                 delta=200.)

  0%|          | 0/5 [00:00<?, ?it/s]Initializing NUTS using jitter+adapt_diag...
/local/home/ib286534/Desktop/wlclusters/.venv/lib/python3.12/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [log10cdelt, log10mdelt]


Output()

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 7 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
 20%|██        | 1/5 [00:13<00:53, 13.29s/it]Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [log10cdelt, log10mdelt]


Output()

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 8 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
 40%|████      | 2/5 [00:24<00:36, 12.29s/it]Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [log10cdelt, log10mdelt]


...

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 8 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
 60%|██████    | 3/5 [00:38<00:25, 12.74s/it]Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [log10cdelt, log10mdelt]


Output()

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 8 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
 80%|████████  | 4/5 [00:49<00:12, 12.11s/it]Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [log10cdelt, log10mdelt]


Output()

Sampling 2 chains for 1_000 tune and 2_000 draw iterations (2_000 + 4_000 draws total) took 8 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
100%|██████████| 5/5 [01:00<00:00, 12.05s/it]


In [10]:
# Ajoute une colonne pour la masse et remplit les masses estimées
wl_cat.add_column(np.full_like(wl_cat['RA'], np.nan), name='M200')
wl_cat['M200'][results["ID"]] = results["m200_med"]

In [11]:
wl_cat[np.isfinite(wl_cat['M200'])]

ID,RA,Dec,SNR,z_p,M200
int64,float64,float64,float64,float64,float64
10,67.4560546875,-30.88698691935369,7.182352267767973,0.2208,106726613180433.25
13,74.37744140625,-31.65071378660751,6.1400448900624705,0.1285,103686999323903.88
18,68.26904296875,-32.70846106177993,6.201005780628554,0.1217,103253937866619.28
22,70.4443359375,-33.21974566949238,5.350307166341964,0.1726,102210078649188.44
23,69.08203125,-33.26434588909905,4.7184645046356515,0.4098,103191917775910.03
